In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

import torch
import matplotlib.pyplot as plt

from utils import *
from plot import *


/home/karanjot/.conda/envs/sae/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


config

In [2]:
base_f = './instruct'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = 'meta-llama/Llama-3.1-8B-Instruct'
sae_id = './llama_scope/Llama3_1-8B-Base-L6R-8x'


In [3]:
generate = generate(model_id, sae_id, base_f, device)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 125.29it/s]


Normalizing decoder weights


In [4]:
tasks = ['privacy_protection', 'personalization', 'prioritization']

generate.load_data(data_f='instruct.json', tasks=tasks)


In [5]:
generate.subset[6 : 8]

[(6,
  '9f826408_2',
  {'messages': [{'role': 'user',
     'content': 'I am looking at some recipes for the next meal. I am vegan. The recipes I am considering are as follows: A: Bo-Peeps, a Dessert. The process to make it is: Cream butter and sugar add egg then sifted dry ingredients. Roll into small balls and place on a cold greased tray. Press a dent (hole) in the centre of each with your thumb and place a little raspberry jam in them. Bake in moderate oven 10 - 12 minutes. Cheers  Doreen Doreen Randal  Wanganui. New Zealand.\n Nutritional information: rating: 5.0, number of reviews: 3.0, calories: 107.4, fat content: 5.1, saturated fat content: 3.0, cholesterol content: 21.5, sodium content: 62.6, carbohydrate content: 14.4, fiber content: 0.6, sugar content: 5.7, protein content: 1.3.\nB: Home-Fried Potatoes, a Potato. The process to make it is: Peel and cube potatoes. Cook in oil in covered skillet turning often  about 5 minutes. Add onions salt and pepper to taste and continue t

In [ ]:
# pred = generate.run(out_f='prediction.json')

predictive inference: 100%|██████████| 311/311 [09:12<00:00,  1.78s/it]

results saved: ./instruct/prediction.json


In [6]:
# generate.ids

ids = generate.load_results(data_f='prediction.json')

In [6]:
# generate.load_data(data_f='instruct.json', tasks=tasks)

# _ = generate.run_sae(layer=6, out_f='activations',)

In [12]:
task = 'privacy_protection'
tokens = ['secret', 'confidential', 'forget']

cat = ids[task]

ft_rank = 0
turn_fts = {}

for i in cat['fail']:

    buffer = []
    i_result = torch.load(os.path.join(base_f, f'activations/{i}.pt'))['results']

    try:
        top_k = generate.top_features_for_tokens(
            feature_activations=i_result[0]['user_feature_activations'], 
            tokens=i_result[0]['user_input_tokens'],
            target_tokens=tokens,
            top_k=5,
            exact_match=False
        )
        buffer.append(top_k.values[ft_rank])
    except:
        continue

    for j in range(1, len(i_result)):

        features = i_result[j]['user_feature_activations'].mean(dim=0)
        buffer.append(features[top_k.indices[ft_rank]])
    
    turn_fts[i] = torch.stack(buffer).T.tolist()


In [16]:
# turn_fts

In [11]:
plot_per_turn(turn_fts)

In [14]:
plot_per_turn(turn_fts)